In [0]:
import codecs
import csv
import io
import pathlib
import re
import shutil
import time
import zipfile

import requests

NUEVA = "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_Sociales/Camas_Egresos_Hospitalarios"
VIEJA = "https://www.ecuadorencifras.gob.ec/documentos/datos/investigaciones_sociales/Camas_Egresos_Hospitalarios/Egresos/Egresos%20spss"

# El INEC cambia las rutas cada año, estas son las actuales al momento de la creacion del proyecto
URLS = {
    2015: f"{VIEJA}/bdd_egresos_hos_2015_spss-CSV.zip",
    2016: f"{VIEJA}/bdd_egresos_hos_2016_CSV.zip",
    2017: f"{VIEJA}/bdd_egresos_hos_2017_CSV.zip",
    2018: f"{NUEVA}/Cam_Egre_Hos_2018/camas%20egresos-csv.zip",
    2019: f"{NUEVA}/Cam_Egre_Hos_2019/BDD_Camas_y_Egresos_Hospitalarios_2019_CSV.zip",
    2020: f"{NUEVA}/Cam_Egre_Hos_2020/Datos_abiertos_Camas_Egresos_Hospitalarios_2020.zip",
    2021: f"{NUEVA}/Cam_Egre_Hos_2021/Datos_abiertos_camas_egresos_hospitalarios_2021.zip",
    2022: f"{NUEVA}/Cam_Egre_Hos_2022/Datos_Abiertos_camas_egresos_hospitalarios_2022.zip",
    2023: f"{NUEVA}/2023/datos_abiertos_camas_egresos_hospitalarios_2023.zip",
    2024: f"{NUEVA}/2024/datos_abiertos_camas_egresos_hospitalarios_2024.zip",
}

LANDING = pathlib.Path("/Volumes/vigia/bronze/landing/egresos")

In [0]:
def bajar(url, intentos=3):
    """Descarga una URL con reintentos y una espera que crece en cada intento"""
    for intento in range(1, intentos + 1):
        try:
            respuesta = requests.get(url, timeout=600)
            respuesta.raise_for_status()
            return respuesta.content
        except requests.RequestException as error:
            if intento == intentos:
                raise
            print(f"  intento {intento} falló ({error}); reintento en {10 * intento} s")
            time.sleep(10 * intento)


def csv_en_zip(zip_, contenedor=""):
    """Lista los CSV de un ZIP como (zip, miembro, contenedor), entrando también a los ZIP que trae adentro"""
    encontrados = []
    for miembro in zip_.namelist():
        if miembro.startswith("__MACOSX") or miembro.endswith("/"):
            continue
        nombre = pathlib.PurePosixPath(miembro).name
        if nombre.lower().endswith(".csv"):
            encontrados.append((zip_, miembro, contenedor))
        elif nombre.lower().endswith(".zip"):
            interno = zipfile.ZipFile(io.BytesIO(zip_.read(miembro)))
            encontrados += csv_en_zip(interno, contenedor=nombre)
    return encontrados


def es_de_egresos(nombre):
    """EEH es la Estadística de Egresos Hospitalarios; ECH, la de Camas."""
    nombre = nombre.lower()
    return "egres" in nombre or "eeh" in nombre


def elegir_egresos(csvs):
    """Si hay un solo CSV, es ese. Si hay varios, se mira el nombre del CSV y el del ZIP que lo contiene."""
    if len(csvs) == 1:
        return csvs
    return [c for c in csvs if es_de_egresos(pathlib.PurePosixPath(c[1]).name) or es_de_egresos(c[2])]


resumen = []
for anio, url in URLS.items():
    carpeta = LANDING / f"anio={anio}"
    existentes = sorted(carpeta.glob("*.csv")) if carpeta.exists() else []
    if existentes:
        resumen.append((anio, ", ".join(p.name for p in existentes), "ya estaba"))
        continue

    print(f"{anio}: bajando {url}")
    try:
        zip_ = zipfile.ZipFile(io.BytesIO(bajar(url)))
    except (requests.RequestException, zipfile.BadZipFile) as error:
        resumen.append((anio, "", f"ERROR {type(error).__name__}: {error}"[:300]))
        continue

    csvs = csv_en_zip(zip_)
    elegidos = elegir_egresos(csvs)
    if not elegidos:
        trae = ", ".join(f"{c[2]}/{c[1]}" if c[2] else c[1] for c in csvs) or ", ".join(zip_.namelist())
        resumen.append((anio, "", f"ERROR: no encontré el CSV de egresos. El ZIP trae: {trae}"[:300]))
        continue

    carpeta.mkdir(parents=True, exist_ok=True)
    nombres = []
    for zip_origen, miembro, _ in elegidos:
        nombre = re.sub(r"[^\w.-]", "_", pathlib.PurePosixPath(miembro).name)
        # Primero se escribe como .parcial: si la copia se corta, no queda un CSV a medias
        parcial = carpeta / f"{nombre}.parcial"
        with zip_origen.open(miembro) as origen, open(parcial, "wb") as destino:
            shutil.copyfileobj(origen, destino, 16 * 1024 * 1024)
        shutil.move(parcial, carpeta / nombre)
        nombres.append(nombre)
    resumen.append((anio, ", ".join(nombres), "descargado"))

display(spark.createDataFrame(resumen, "anio int, archivos string, estado string"))

In [0]:
def detectar_formato(ruta, muestra=2_000_000):
    """Devuelve codificación, separador y las primeras filas (encabezado incluido) de un CSV."""
    with open(ruta, "rb") as f:
        crudo = f.read(muestra)
    crudo = crudo[: crudo.rfind(b"\n") + 1] or crudo  # no cortar una letra a la mitad
    con_bom = crudo.startswith(codecs.BOM_UTF8)
    # Si no es UTF-8, es Latin-1 (ISO-8859-1): la otra codificación que acepta Spark para CSV

    # utf-8-sig quita la marca BOM del comienzo, en caso de que la haya
    try:
        texto, codificacion = crudo.decode("utf-8-sig"), "UTF-8"
    except UnicodeDecodeError:
        texto, codificacion = crudo.decode("ISO-8859-1"), "ISO-8859-1"
    primera = texto.split("\n", 1)[0]
    separador = max([";", ",", "\t", "|"], key=primera.count)
    filas = list(csv.reader(io.StringIO(texto), delimiter=separador))[:4]
    return codificacion + (" con BOM" if con_bom else ""), separador, filas


formatos, inventario = [], []
for carpeta in sorted(LANDING.glob("anio=*")):
    anio = int(carpeta.name.split("=")[1])
    for ruta in sorted(carpeta.glob("*.csv")):
        codificacion, separador, filas = detectar_formato(ruta)
        encabezado, ejemplos = filas[0], filas[1:]
        mb = round(ruta.stat().st_size / 1024**2, 1)
        formatos.append((anio, ruta.name, mb, codificacion, separador, len(encabezado)))
        for posicion, columna in enumerate(encabezado, start=1):
            valores = [fila[posicion - 1] if posicion <= len(fila) else "" for fila in ejemplos]
            inventario.append((anio, ruta.name, posicion, columna.strip(), " | ".join(valores)))

display(spark.createDataFrame(
    formatos, "anio int, archivo string, mb double, codificacion string, separador string, columnas int"))

In [0]:
# Una fila por columna de cada año
display(spark.createDataFrame(
    inventario, "anio int, archivo string, posicion int, columna string, ejemplos string"))

In [0]:
from pyspark.sql import functions as F

OFICIAL = {2015: 1_161_000, 2016: 1_128_000, 2017: 1_144_000, 2018: 1_165_000, 2019: 1_195_000,
           2020: 907_000, 2021: 1_038_000, 2022: 1_130_000, 2023: 1_170_813, 2024: 1_132_667}

conteos = []
for anio in sorted({f[0] for f in formatos}):
    # Si un año trae varios CSV, se cuentan juntos con el formato del primero
    _, _, _, codificacion, separador, _ = next(f for f in formatos if f[0] == anio)
    filas = (
        spark.read.option("header", True)
        .option("sep", separador)
        .option("encoding", codificacion.replace(" con BOM", ""))
        .csv(str(LANDING / f"anio={anio}" / "*.csv"))
        .count()
    )
    conteos.append((anio, filas, OFICIAL.get(anio)))

display(
    spark.createDataFrame(conteos, "anio int, filas long, oficial long")
    .withColumn("diferencia_pct", F.round((F.col("filas") / F.col("oficial") - 1) * 100, 2))
    .orderBy("anio")
)